# Member 2 — Conventional LAOS Analysis (Final)

**Purpose.** Validate conventional LAOS measures and test whether they describe strain-dependent nonlinear behaviour in the shared Armstrong blood dataset.

**Final freeze:** 181 analysis-ready LAOS measurements from 10 source workbooks. The exploratory count of 184 is not used for the final analysis; see `dataset_freeze_audit.csv`.

## Analysis decisions

- Use the Member 1 standardized signals and metadata.
- Exclude the 17 duplicated 9MAY19 LAOS blocks, the exact NOV18 duplicate trace, and the unlabeled JUN18 block.
- Do **not** manually reinsert the malformed JUN18 ω=1, γ₀=10 source block, because it was not standardized by the shared pipeline.
- Primary feature estimate uses the last two nominal cycles.
- No per-window z-score normalization.
- Treat source workbook as the independent unit; repeated storage conditions are collapsed before workbook-level strain-amplitude correlations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats, signal
import matplotlib.pyplot as plt

# Point this to the team Member 1 preprocessing folder.
DATA_DIR = Path('../0924dataprocessing')
if not DATA_DIR.exists():
    DATA_DIR = Path('/mnt/data/p1/0924dataprocessing')

measurement_index = pd.read_csv(DATA_DIR/'measurement_index.csv')
signals = pd.read_csv(DATA_DIR/'standardized_signals.csv')
laos = measurement_index[measurement_index.experiment_type.eq('LAOS')].copy()

exclude = (
    laos.source_file.eq('9MAY19 UDEL Blood.xlsx')
    | laos.duplicate_of_measurement_id.notna()
    | laos.omega_label_rad_s.isna()
    | laos.strain_amplitude_label_fraction.isna()
)
laos_final = laos.loc[~exclude].copy()
print('Final LAOS freeze:', len(laos_final), 'measurements from', laos_final.source_file.nunique(), 'workbooks')

## Conventional LAOS feature extraction

The measured strain fundamental establishes phase. Stress is represented with sine/cosine terms for harmonics 1–5. G′ and G″ come from the first-harmonic in-phase and quadrature components divided by the measured strain fundamental amplitude. Harmonic ratios use the magnitude of the sine/cosine pairs. This is consistent with the Fourier definition used in Armstrong et al. (2021) and Pincot et al. (2023).

In [ ]:
def fit_metrics(d, omega, use_last_cycles=2, detrend=False):
    d = d[['time_s','stress_pa','strain_pct']].dropna().sort_values('time_s').copy()
    t = d.time_s.to_numpy(float)
    s = d.stress_pa.to_numpy(float)
    g = d.strain_pct.to_numpy(float)/100.0
    if len(t) < 20 or not np.isfinite(omega) or omega <= 0:
        return None

    T = 2*np.pi/omega
    tstart = t.max() - use_last_cycles*T
    keep = t >= tstart
    if keep.sum() < 20:
        keep = np.ones(len(t), dtype=bool)
    t, s, g = t[keep], s[keep], g[keep]
    if detrend:
        s = signal.detrend(s)
        g = signal.detrend(g)

    Xg = np.column_stack([np.sin(omega*t), np.cos(omega*t), np.ones_like(t)])
    cg = np.linalg.lstsq(Xg, g, rcond=None)[0]
    gamma1 = np.hypot(cg[0], cg[1])
    phi = np.arctan2(cg[1], cg[0])
    theta = omega*t + phi

    cols = []
    for n in range(1,6):
        cols += [np.sin(n*theta), np.cos(n*theta)]
    X = np.column_stack(cols + [np.ones_like(t)])
    c = np.linalg.lstsq(X, s, rcond=None)[0]
    sincoef = np.array([c[2*i] for i in range(5)])
    coscoef = np.array([c[2*i+1] for i in range(5)])
    amp = np.hypot(sincoef, coscoef)

    Gp = sincoef[0]/gamma1
    Gpp = coscoef[0]/gamma1
    delta = np.degrees(np.arctan2(Gpp, Gp))

    last = t >= t.max()-T
    area = abs(np.trapz(s[last], g[last]))
    denom = np.ptp(s[last])*np.ptp(g[last])

    pred = X@c
    r2 = 1 - np.sum((s-pred)**2)/np.sum((s-s.mean())**2)
    return {
        'G_prime_Pa': Gp,
        'G_double_prime_Pa': Gpp,
        'phase_angle_deg': delta,
        'I1_Pa': amp[0],
        'I3_Pa': amp[2],
        'I5_Pa': amp[4],
        'I3_I1': amp[2]/amp[0],
        'I5_I1': amp[4]/amp[0],
        'THD_2to5': np.sqrt(np.sum(amp[1:]**2))/amp[0],
        'lissajous_normalized_area': area/denom if denom > 0 else np.nan,
        'harmonic_fit_R2': r2,
    }

In [ ]:
rows = []
for _, r in laos_final.iterrows():
    d = signals[signals.measurement_id.eq(r.measurement_id)]
    m = fit_metrics(d, float(r.omega_label_rad_s), use_last_cycles=2, detrend=False)
    if m is None:
        continue
    rows.append({
        'measurement_id': r.measurement_id,
        'source_file': r.source_file,
        'sheet': r.sheet,
        'omega_rad_s': r.omega_label_rad_s,
        'strain_amplitude': r.strain_amplitude_label_fraction,
        **m
    })
features = pd.DataFrame(rows)
features.head()

## Final statistical summary

Repeated measurements at the same strain amplitude/frequency within a workbook are first collapsed by median (important for DEC17 0/12/24 h). Spearman correlation is then calculated within each workbook. This avoids pseudoreplication from treating cycles or storage repeats as independent biological samples.

In [ ]:
metrics = ['G_prime_Pa','G_double_prime_Pa','phase_angle_deg','I3_I1','I5_I1','THD_2to5','lissajous_normalized_area']
summary = []
for omega, g in features.groupby('omega_rad_s'):
    for metric in metrics:
        rhos = []
        for workbook, gg in g.groupby('source_file'):
            a = gg.groupby('strain_amplitude', as_index=False)[metric].median().dropna().sort_values('strain_amplitude')
            if len(a) >= 4:
                rho, p = stats.spearmanr(np.log10(a.strain_amplitude), a[metric])
                rhos.append(rho)
        if rhos:
            summary.append([omega, metric, len(rhos), np.median(rhos)])
pd.DataFrame(summary, columns=['omega_rad_s','metric','n_workbooks','median_spearman_rho'])

## Instrument cross-validation and final outputs

The submission package includes `instrument_crossvalidation.csv` and `instrument_crossvalidation_summary.csv`, generated by matching Armstrong SS/SAOS frequency-sweep rows to LAOS conditions. The final report uses 26 matched G′/G″ pairs and 12 matched I₃/I₁ pairs.

Use the files in `figures/` for the final report. `Member2_Conventional_LAOS_Methods_Results.docx` contains the submission-ready prose and selected figures.